# Урок 8.5. Инфраструктура и тестирование

Сервис, который нельзя протестировать без GPU и трёх запущенных серверов, —
это сервис, который никто не тестирует. В этом уроке — как проект остаётся
проверяемым и наблюдаемым: тесты на фейках, health-пробы, метрики, Docker.

🎯 **Цели урока:**

- Прогнать все тесты проекта без Qdrant, Ollama и моделей — и понять, почему
  это возможно.
- Разобрать паттерн «фейки на границе app.state» и тесты НЕ-вызовов.
- Увидеть, как Pydantic-схемы держат контракт API: строгий разбор, правила
  на несколько полей, вычисляемые поля, типизированные SSE-события.
- Живьём увидеть деградацию `/ready` при выключенной зависимости.
- Пройтись по метрикам (включая ловушку с 307-редиректом) и Docker-упаковке.

## 1. Тесты: фейки на границе

Стратегия из [`tests/conftest.py`](project/tests/conftest.py) — та же, что
в модулях 3 и 5, доведённая до канона:

- **Граница подмены — `app.state`.** Приложение получает сервисы из state
  (кладёт их туда lifespan), тесты кладут туда фейки. Ни monkeypatch по путям
  импорта, ни importlib-магии — обычное присваивание.
- **`environment="test"` пропускает тяжёлый lifespan**: модели не грузятся,
  сеть не трогается. Настройка приоритетов pydantic-settings из урока 8.2
  делает это одной строчкой.
- **Фейки со счётчиками**: `FakeRag` записывает вызовы, `FakeHistory` —
  сохранения. Это позволяет тестировать и **НЕ-вызовы** — например, что
  при 422 сервис не пошёл в RAG (привычка из урока 3.8: самые дорогие баги —
  лишние вызовы).
- Уровни пирамиды: юнит (чанкинг, промпт, uuid5, конфиг, схемы — чистые функции),
  интеграционные (HTTP через TestClient с фейками), и **SQLite тестируется
  по-настоящему** — во временном файле: фейкать хранилище, которое и так
  embedded, нет смысла.

Запускаем всё:

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-X", "utf8", "-m", "pytest", "-v", "--no-header", "-o", "addopts="],
    cwd="project", capture_output=True, text=True, encoding="utf-8", timeout=300,
)
lines = result.stdout.splitlines()
for line in lines:
    if "::" in line or "passed" in line or "failed" in line:
        print(line)

**73 теста за секунду** — без GPU, без скачанных моделей, без единого
запущенного сервера. Это не «tests lite»: покрыты контракты обоих эндпоинтов
(включая 422 на мусор и 503 без сервисов), полный протокол SSE-стрима,
сохранение истории в обоих режимах, чанкинг со всеми крайностями, приоритеты
конфига и настоящий SQLite во временном файле — а схемы API проверяются
отдельным слоем из 29 юнит-тестов вообще без HTTP (о них — следующий раздел).

Такой набор запускается на каждый коммит не потому, что «так надо», а потому,
что он ничего не стоит — в этом и был замысел границы app.state.

## 2. Контракт API: схемы, которые умеют больше

В уроках 8.3–8.4 схемы из [`app/schemas.py`](project/app/schemas.py) были
«формой»: поля и границы значений. Форма пропускает три класса тихих ошибок,
и все три встречаются в чужих (и своих) RAG-сервисах:

- **опечатка в имени поля** — `topk` вместо `top_k` молча игнорируется, клиент
  получает пять результатов вместо трёх и не понимает почему;
- **«догадливые» приведения типов** — `"3"` становится 3, `"yes"` — True.
  До того дня, когда клиент пришлёт `"no"` и тоже получит True;
- **факты, которые каждый клиент считает сам** — какие `[N]` процитированы,
  не выдуман ли номер, отказ ли это. Помните живой отказ урока 8.4 —
  «В документации ответа не нашёл **[1]-[5]**»? Наивный клиент видит в нём ответ.

Схемы v2 закрывают это средствами самого Pydantic — без новой логики в роутерах:

| Приём | Где | Что даёт |
|---|---|---|
| `ConfigDict(extra="forbid", strict=True, str_strip_whitespace=True)` | база `ApiRequest` | опечатки, числа строкой, пробелы вместо текста — 422 |
| `Annotated[str, StringConstraints(...)]` | `QueryText`, `SourceName`, `TopK`, … | одно определение поля — один смысл во всех схемах |
| `@model_validator(mode="after")` | `SearchRequest` | правила на несколько полей: `fetch_k >= top_k`; `fetch_k` только с reranker |
| `@computed_field` | `SearchResponse`, `GenerateResponse`, `ReadyResponse` | `count`, `ranked_by`, `cited`, `dangling_citations`, `is_refusal`, `failed` |
| discriminated union по `event` | `SourcesEvent \| TokenEvent \| DoneEvent \| ErrorEvent` | протокол SSE в типах; `parse_sse` для клиентов и тестов |
| `@field_serializer` / `@field_validator` | `score`, `created_at` | четыре знака на проводе; ISO 8601 с явным UTC |
| инварианты в `model_validator` | `ReadyResponse`, `GenerateResponse` | `ready` ⇔ все чеки ok; источники пронумерованы 1..N |
| `ErrorResponse` + обработчики исключений | `app/main.py` | один формат ошибок `{detail, request_id}` для 5xx/503/404 |

Решение и его цена записаны в [ADR-0005](project/docs/adr/0005-api-contract.md).
Проверим всё это без серверов — на том же TestClient с фейками, что и в тестах.


In [ ]:
import json
import logging
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path("project").resolve()))
os.environ.setdefault("APP_ENVIRONMENT", "test")

from fastapi.testclient import TestClient

from app.config import Settings
from app.main import create_app
from tests.conftest import FakeHistory, FakeRag

# Тот же приём, что в conftest: приложение с фейками на границе app.state.
fake_rag, fake_history = FakeRag(), FakeHistory()
app = create_app(Settings(_env_file=None, environment="test"))
app.state.rag, app.state.history = fake_rag, fake_history
client = TestClient(app)
logging.getLogger("app.access").setLevel(logging.WARNING)   # access-лог здесь только шумит

BAD_REQUESTS = [
    ("/api/search",   {"query": "   "},                                       "пробелы вместо запроса"),
    ("/api/search",   {"query": "деплой", "topk": 3},                         "опечатка в имени поля"),
    ("/api/search",   {"query": "деплой", "top_k": "3"},                      "число строкой"),
    ("/api/generate", {"query": "деплой", "stream": "yes"},                   "булево строкой"),
    ("/api/search",   {"query": "деплой", "top_k": 5, "fetch_k": 3},          "сеть уже итога"),
    ("/api/search",   {"query": "деплой", "fetch_k": 30, "use_reranker": False}, "fetch_k без reranker"),
    ("/api/search",   {"query": "деплой", "source": "../etc/passwd"},         "путь вместо имени файла"),
]
for path, payload, label in BAD_REQUESTS:
    response = client.post(path, json=payload)
    error = response.json()["detail"][0]
    loc = ".".join(str(part) for part in error["loc"] if part != "body") or "<тело целиком>"
    print(f"{response.status_code}  {label:24} {loc:15} {error['msg']}")

print("\nвызовов RagService после семи запросов:", len(fake_rag.search_calls) + len(fake_rag.answer_calls))

Семь запросов — семь 422 и **ноль вызовов RagService**: мусор не дошёл даже до
эмбеддера. Три вещи в выводе (мой прогон), на которые стоит посмотреть:

- **Три уровня правил в одном формате ошибки.** Ограничения поля (`query`
  слишком короткий *после* обрезки пробелов), строгость типов (`top_k` — не
  integer, `stream` — не boolean, `topk` — «Extra inputs are not permitted»)
  и правила на несколько полей. У последних `loc` пустой (`<тело целиком>`),
  потому что виновато не поле, а сочетание: «fetch_k (3) должен быть не меньше
  top_k (5)». Текст — наш, из `ValueError` в `model_validator`, и он говорит,
  что именно исправить.
- **Ни строки новой логики в роутере.** `routers/search.py` ради этих проверок
  не изменился — всё живёт в схеме и отрабатывает до вызова обработчика.
  Роутеру достаётся только валидное тело.
- **Цена — миллисекунды** против ~590 мс поиска с reranker'ом из урока 8.3.
  Строгая валидация — самая дешёвая защита пайплайна из всех, что мы строили
  в курсе, а дороже всего стоят как раз *лишние* вызовы (урок 3.8).

Регулярка для `source` (`^[\w][\w.\-]*\.md$`) — не про безопасность файловой
системы (это payload-фильтр Qdrant, а не путь на диске), а про честность
контракта: `../etc/passwd` в запросе означает, что клиент перепутал поле, и
лучше сказать ему об этом сразу, чем вернуть пустую выдачу.


In [ ]:
from pydantic import ValidationError

from app.schemas import DependencyStatus, GenerateResponse, HistoryItem, ReadyResponse, SourceRef


def make_refs(count: int) -> list[SourceRef]:
    return [SourceRef(number=i, source=f"doc-{i}.md", title=f"Документ {i}", chunk_index=0, score=0.8)
            for i in range(1, count + 1)]


ANSWERS = [   # (текст ответа, сколько источников было в промпте)
    ("Ротация важнее вычистки: секрет уже скомпрометирован [1]. Историю чистим потом [2].", 2),
    ("Деплой запускается по тегу [1], откат — тем же job [7].", 2),
    ("В документации ответа не нашёл [1]-[5].", 5),     # живой ответ qwen2.5:3b из урока 8.4
]
for answer, n_sources in ANSWERS:
    resp = GenerateResponse(answer=answer, sources=make_refs(n_sources), model="demo", took_ms=1)
    print(f"cited={resp.cited!s:7} dangling={resp.dangling_citations!s:4} refusal={resp.is_refusal!s:5}  «{answer[:46]}…»")

print()
checks = [DependencyStatus(name="qdrant", ok=True),
          DependencyStatus(name="ollama", ok=False, detail="сервер не отвечает")]
ready = ReadyResponse.from_checks(checks)
print("from_checks         ->", ready.status, "| failed =", ready.failed)
try:
    ReadyResponse(status="ready", checks=checks)
except ValidationError as exc:
    print("ready + упавший чек ->", exc.errors()[0]["msg"])

item = HistoryItem(id=1, created_at="2026-08-28 13:53:12", question="?", answer="!",
                   sources=[], model="m", took_ms=1)
print("\nSQLite отдал '2026-08-28 13:53:12' -> клиент получит", item.model_dump(mode="json")["created_at"])

Ответы теперь несут не только данные, но и выводы из них — одинаковые для
любого клиента:

- **`cited` / `dangling_citations`.** Первый ответ ссылается на [1] и [2] — оба
  источника существуют, `dangling` пуст. Во втором модель «придумала» [7] при
  двух источниках — `dangling_citations=[7]`: UI может подсветить такую ссылку
  красным, а eval-пайплайн (урок 3.3) — считать долю ответов с висячими
  ссылками как метрику качества. Раньше это умел только тот, кто написал
  регулярку у себя.
- **`is_refusal`.** Живой отказ из урока 8.4 — «В документации ответа не нашёл
  [1]-[5]» — распознан как отказ **по вхождению** канонической фразы, несмотря
  на хвост из ссылок; при пяти источниках в промпте все ссылки существуют,
  `dangling` пуст. Константа `REFUSAL_MARKER` одна и для промпта, и для
  детектора: поменяете формулировку отказа — детектор поменяется вместе с ней.
- **Инвариант `/ready`.** `from_checks` вычислил `degraded` и список виновников
  `failed=['ollama']`; попытка собрать `ready` с упавшим чеком — ValidationError
  с внятным текстом. Квиз этого урока утверждал, что статус и чеки «не могут
  разойтись»; теперь это гарантирует не дисциплина автора, а тип.
- **`created_at`.** Наивная строка SQLite превратилась в `2026-08-28T13:53:12Z` —
  ISO 8601 с явным UTC. В живом выводе урока 8.4 вы видели ещё старый формат:
  тот прогон был до этого изменения — так и выглядит эволюция контракта, и
  именно поэтому она записана в ADR-0005, а не только в коде.


In [ ]:
from app.schemas import TokenEvent, parse_sse

typed = []
with client.stream("POST", "/api/generate", json={"query": "Как деплоить?", "stream": True}) as response:
    name = None
    for line in response.iter_lines():
        if line.startswith("event:"):
            name = line.split(" ", 1)[1]
        elif line.startswith("data:") and name:
            typed.append(parse_sse(name, line.split(" ", 1)[1]))
            name = None

print("типы событий:", [type(event).__name__ for event in typed])
print("источники первого события:", [ref.source for ref in typed[0].sources])
print("текст из токенов:", "".join(event.text for event in typed if isinstance(event, TokenEvent)))
print("финал:", typed[-1])

try:
    parse_sse("boom", '{"text": "x"}')
except ValidationError as exc:
    print("\nнеизвестное событие ->", exc.errors()[0]["msg"])


Тот же поток, что в уроке 8.4 и в `test_generate_stream_sse`, — но каждое
событие теперь **объект известного типа**: `SourcesEvent` → пять `TokenEvent` →
`DoneEvent`. `parse_sse` берёт имя из строки `event:` и JSON из `data:`,
а discriminated union по полю `event` выбирает модель. Неизвестное событие
`boom` — ValidationError, в тексте которой перечислены все допустимые имена:
протокол задокументирован самой ошибкой.

Формат на проводе не изменился ни на байт (имя — в `event:`, поля — в `data:`,
разделитель `\n\n` — внутри `format_sse`), поэтому старые клиенты и тест из
раздела 3 работают как прежде. Изменилось то, что роутер собирает события из
моделей, а не из словарей: опечатка в ключе вроде `"tokn"` теперь ловится
на этапе `TokenEvent(text=...)`, а не в браузере пользователя.


In [ ]:
spec = app.openapi()
schemas = spec["components"]["schemas"]
print("SearchRequest.additionalProperties:", schemas["SearchRequest"]["additionalProperties"])
print("SearchRequest.examples[0]:", schemas["SearchRequest"]["examples"][0])
print("SearchResponse.required:", schemas["SearchResponse"]["required"])
print("ranked_by:", {k: schemas["SearchResponse"]["properties"]["ranked_by"][k] for k in ("enum", "readOnly")})
generate = spec["paths"]["/api/generate"]["post"]["responses"]
print("generate 200 media types:", list(generate["200"]["content"]))
print("generate коды ошибок:", [code for code in generate if code != "200"],
      "| 502 ->", generate["502"]["content"]["application/json"]["schema"]["$ref"])


Всё это бесплатно попадает в OpenAPI — то есть в `/docs` и в генераторы
клиентов:

- `additionalProperties: false` у `SearchRequest` — генератор SDK не даст
  собрать запрос с лишним полем;
- `examples` из `json_schema_extra` — кнопка «Try it out» в Swagger заполнена
  осмысленным запросом, а не `"string"`;
- computed-поля в `required` с пометкой `readOnly: true` — клиент знает, что
  `count` и `ranked_by` всегда придут, но отправлять их не нужно; у `ranked_by`
  виден `enum` из трёх значений;
- у `/api/generate` два media type в 200 (`application/json` и
  `text/event-stream`) и задокументированные `502`/`503` со ссылкой на
  `ErrorResponse` — потребитель видит форму ошибки до первого инцидента.

Контракт стал исполняемым в обе стороны: схема отвергает мусор на входе,
считает выводы на выходе и описывает сама себя в документации. А то, что она
обещает, сторожат тесты — к самому важному из них и переходим.


## 3. Самый важный тест: контракт стрима

Отдельно посмотрите на `test_generate_stream_sse`
([tests/integration/test_generate_api.py](project/tests/integration/test_generate_api.py)) —
он фиксирует протокол урока 8.4 как исполняемую спецификацию:

```python
events = [...]                       # распарсенные event: из потока
assert events[0] == "sources"        # источники - ДО первого токена
assert events[-1] == "done"          # финал с метаданными
assert events[1:-1] == ["token"] * len(STREAM_TOKENS)
typed = [parse_sse(name, data) for name, data in zip(events, datas)]  # типизируем

assert fake_history.saved[0]["answer"] == "".join(STREAM_TOKENS)   # история!
```

Если кто-то «оптимизирует» роутер и источники поедут после токенов — упадёт
тест, а не прод. Контракты, о которых договорились словами, живут до первого
рефакторинга; контракты, записанные тестами, — пока живут тесты.

## 4. Health-пробы: живой и готовый

Из урока 5.5 (пирамида health-check) и документа `11-observability.md` нашей
же базы знаний: **`/health` без внешних проверок** (иначе падение Qdrant
заставит оркестратор рестартить здоровый процесс — каскад), **`/ready`
с проверками зависимостей** (по нему решают, слать ли трафик).

Проверим деградацию по-настоящему: поднимем сервис с адресом Ollama,
указывающим в пустоту.

In [ ]:
import json
import os
import time

import httpx

BASE = "http://127.0.0.1:8011"
env = {**os.environ, "APP_OLLAMA_BASE_URL": "http://127.0.0.1:9999"}   # мёртвый порт

server = subprocess.Popen(
    [sys.executable, "-X", "utf8", "-m", "uvicorn", "app.main:app",
     "--host", "127.0.0.1", "--port", "8011"],
    cwd="project", env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
with httpx.Client(timeout=10) as probe:
    for attempt in range(240):
        try:
            if probe.get(f"{BASE}/health").status_code == 200:
                break
        except httpx.HTTPError:
            pass
        if server.poll() is not None:
            raise RuntimeError("uvicorn умер при старте")
        time.sleep(0.5)

    health = probe.get(f"{BASE}/health")
    ready = probe.get(f"{BASE}/ready")
    print("/health:", health.status_code, health.json())
    print("/ready: ", ready.status_code, json.dumps(ready.json(), ensure_ascii=False))

Ровно то, что обещали пробам в уроке 5.5:

- **`/health`: 200** — процесс жив, и это правда: он поднялся, загрузил
  эмбеддер, отвечает на HTTP. Оркестратору незачем его перезапускать —
  рестарт Ollama не починит.
- **`/ready`: 503** с точным диагнозом: `qdrant: ok, ollama: false — сервер
  не отвечает`. Балансировщик по коду понимает «трафик не слать», дежурный
  по телу — «чинить Ollama, Qdrant не трогать».

Заметьте, как дёшево нам далась эта проверка: другой адрес в переменной
окружения — и никакого «мока даунтайма». Конфигурация через окружение
(урок 8.2) окупается и в тестировании отказов.

## 5. Метрики и одна ловушка

Сервис отдаёт Prometheus-метрики: счётчик запросов по методу/пути/статусу,
гистограммы `rag_search_duration_seconds` и `rag_generation_duration_seconds`
(с бакетами под медленную LLM — до 240 секунд, стандартные «до 10 с» здесь
бесполезны), счётчики токенов и ошибок LLM.

Ловушка, которую мы поймали при живой проверке: `/metrics` смонтирован как
отдельное ASGI-приложение (`app.mount`), и Starlette отвечает на `GET /metrics`
**редиректом 307 на `/metrics/`**. Браузер и Prometheus ходят по редиректам
сами, а вот httpx по умолчанию — нет. Проверим оба поведения на нашем
деградированном сервере (метрикам зависимости не нужны):

In [ ]:
with httpx.Client(timeout=10) as plain:
    print("без follow_redirects:", plain.get(f"{BASE}/metrics").status_code)

with httpx.Client(timeout=10, follow_redirects=True) as follower:
    text = follower.get(f"{BASE}/metrics").text
    wanted = ("http_requests_total{", "rag_search_duration_seconds_count",
              "rag_llm_errors_total")
    for line in text.splitlines():
        if line.startswith(wanted):
            print(line)

server.terminate()
server.wait(timeout=15)
print("\nдеградированный uvicorn остановлен")

Счётчики на месте — обратите внимание, что несколько запросов `/health`
из пробника уже посчитаны с метками метода, пути и статуса. Правило
кардинальности из `11-observability.md` соблюдено конструктивно: у нас нет
путевых параметров, поэтому сырой `request.url.path` безопасен (в сервисе
с `/items/{id}` сюда обязан идти шаблон маршрута, не сырой путь).

## 6. Docker: упаковка

[`Dockerfile`](project/Dockerfile) — три решения:

1. **Слои с зависимостями отдельно от кода**: `COPY requirements.txt` +
   `pip install` кешируются, правка кода не пересобирает 2 ГБ с torch.
2. **CPU-сборка PyTorch** через `--extra-index-url .../whl/cpu` — вдвое легче
   CUDA-версии, которая в контейнере без GPU бесполезна.
3. **Адреса соседей — через окружение**: в compose-сети Qdrant зовут
   `http://qdrant:6333`, а не 127.0.0.1 — тот же конфиг-класс, другие значения.

[`docker-compose.yml`](project/docker-compose.yml) поднимает тройку
qdrant+ollama+app с волюмами (модели Ollama и история переживают пересоздание
контейнеров). Четвёртое решение — **порты**: Ollama наружу не публикуется вовсе
(у неё нет аутентификации, лимитов и потолка `num_predict` — урок 5.7 ставил перед
ней прокси именно поэтому; здесь к ней ходит только `app` по внутренней сети),
Qdrant и приложение привязаны к `127.0.0.1`, а не к `0.0.0.0`. Голая `"11434:11434"`
в compose — это открытая LLM для всей локальной сети. Честная пометка из файла: compose-вариант написан по документации,
но при сборке курса не прогонялся — сервис проверялся локальным uvicorn против
локальных зависимостей. Проверите — расскажите.

## ⚠️ Частые ошибки

1. **Внешние проверки в liveness.** Упал Qdrant — оркестратор перезапускает
   здоровое приложение по кругу. `/health` отвечает про процесс, `/ready` —
   про зависимости.
2. **Тесты, требующие живой Ollama.** Они не гоняются в CI, значит, их нет.
   Фейки на границе + отдельный небольшой e2e-набор по желанию (наши уроки
   8.3–8.4 и есть такой e2e).
3. **Забытый `follow_redirects` при походе в `/metrics`.** 307 вместо метрик
   в самодельном скрейпере — и дашборд пустой.
4. **Гистограммы с дефолтными бакетами для LLM.** Стандартная сетка
   заканчивается на 10 с — вся генерация упадёт в «+Inf», перцентили ослепнут
   (урок 5.5 про пороги, урок 4.3 про «среднее лжёт»).
5. **CUDA-torch в CPU-контейнере.** +3 ГБа образа ради нуля пользы.
6. **`ports: "11434:11434"` у Ollama в compose.** Docker публикует порт на всех
   интерфейсах и в обход файрвола Windows/Linux: любой сосед по Wi-Fi получает
   вашу GPU и модель без токена. Внутренним сервисам порт наружу не нужен —
   они видят друг друга по имени сервиса; хосту — только `127.0.0.1:порт`.
7. **«Догадливая» валидация запросов.** Lax-режим Pydantic превращает `"3"`
   в 3 и молча выбрасывает опечатанные поля — клиент узнаёт об ошибке по
   странным результатам, а не по 422. Строгость на границе дешевле догадок
   внутри (ADR-0005).

## Мини-квиз

**1. Почему SQLite в тестах настоящий, а Qdrant — фейковый?**

<details><summary>Ответ</summary>

Цена и ценность: SQLite embedded — временный файл бесплатен, а тест проверяет
настоящий SQL и сериализацию. Qdrant — отдельный сервер: его подъём в юнит-тестах
дорог, а проверяем мы не Qdrant (он протестирован своими авторами), а НАШУ
логику вокруг него — для неё хватает фейка с контрактом.
</details>

**2. `/ready` вернул 503, но в теле `qdrant: ok, ollama: ok` быть не может. А наоборот — 200 с ollama: false?**

<details><summary>Ответ</summary>

Тоже нет: статус вычисляется из чеков (`all(ok)`), они не могут разойтись.
Это осознанный дизайн: машинам — код 503 («не слать трафик»), людям — тело
с именами виновников. Двойная адресация одного факта.
</details>

**3. Зачем тесту стрима проверять `fake_history.saved`, если история — «побочный эффект»?**

<details><summary>Ответ</summary>

Потому что сохранение в finally — это контракт надёжности (частичный ответ
не теряется при обрыве), а контракты без тестов умирают при рефакторинге.
«Побочные эффекты» с бизнес-ценностью — полноправный предмет тестирования,
как и НЕ-вызовы.
</details>

**4. `is_refusal` можно было посчитать в роутере и положить в обычное поле ответа. Чем `computed_field` лучше?**

<details><summary>Ответ</summary>

У ответа два места сборки (JSON и SSE) плюс тесты и будущие клиенты — логика
в роутере рано или поздно разойдётся между ними. `computed_field` считается
одинаково везде, где есть объект схемы, а константа `REFUSAL_MARKER` общая
с промптом: детектор и текст отказа не могут разъехаться. Контракт живёт
в одном файле и покрыт unit-тестами без HTTP.
</details>

## 🎓 Итоги

- 73 теста за ~секунду без моделей и серверов: фейки на границе app.state,
  `environment=test`, счётчики вызовов и НЕ-вызовов.
- Схемы — контракт, а не форма: строгий разбор (extra=forbid, strict), правила
  на несколько полей, computed-поля (`cited`, `is_refusal`, `ranked_by`,
  `failed`), SSE-события как discriminated union, инварианты ответов, единый
  `ErrorResponse` — всё это средствами Pydantic и покрыто тестами без HTTP.
- Контракт SSE-стрима зафиксирован тестом — спецификация, которая не устаревает.
- `/health` ≠ `/ready`: живой процесс против готовности зависимостей; деградация
  проверена живьём — 503 со списком виновников.
- Метрики: LLM-бакеты до 240 с, кардинальность под контролем, `/metrics`
  отвечает 307 без follow_redirects.
- Docker: кешируемые слои, CPU-torch, адреса через окружение, порты — только
  loopback, Ollama наружу не публикуется (урок 5.7).

**Дальше — урок 8.6, финал:** что мы упростили и почему, как это масштабировать,
и карта всего курса в одном проекте.